# SDA-DSC-212 — Time Series Analysis and Forecasting
## Lab 7 — End-to-End Selection and the Decision Record

**Duration** 50 minutes · **Work in pairs** · `git checkout lab7-solution`

> The one rule for this whole course: every feature, every split and every scaler
> must be computable using only data that existed at the forecast origin.

### Tasks

| min | task |
|---|---|
| 8 | Wire the pipeline end to end from a **committed config** and confirm a clean run reproduces the Lab 6 backtest table. Print `MATCH`. Then read the inherited selection step and find the **fault** in it. |
| 10 | Three constraint scenarios with explicit weights over accuracy / explainability / latency / maintenance. Build the candidate profile table and run `recommend.score` for each. |
| 10 | Observe that the champion **changes** across scenarios. Record which constraint flipped each decision, and use `flip_conditions` to state what would have to change for the runner-up to win. |
| 8 | Generate the reserve brief with `pipeline.brief.reserve_brief`, then critique it as a control-room engineer would. Rewrite the jargon version in decision language. |
| 9 | Write `DECISION_RECORD.md`: the decision and its cost function, the candidates, the constraints and weights, the recommendation with its DM evidence, the flip-conditions, and the monitoring contract. |
| 5 | Commit. |

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})

DATA = "../data"

### Task 1 — the config, the clean run, and the selection step you inherited

**Reproducibility is what makes the Lab 6 report evidence rather than anecdote.** Three things
buy it, and all three are cheap: **fixed splits** (dates in a file, not in a head), **fixed
seeds** (every LightGBM in `src/tayyar` is seeded at 212), and **pinned dependencies** (a
LightGBM minor version can move the third decimal). A number nobody else can regenerate is not
a result; it is a claim.

The test is blunt: run the whole pipeline from the committed config in a clean kernel and
check that it reproduces `reference_results.json` exactly. If it prints `MATCH`, the report
can be defended. If it does not, nothing downstream of it means anything.

In [ ]:
import json

CONFIG = {
    "data": {"demand": f"{DATA}/ksa_grid_demand.csv",
             "calendar": f"{DATA}/ksa_calendar.csv"},
    "features": {"source_columns": ["demand_mw", "temp_c", "is_weekend",
                                    "is_ramadan", "is_eid", "is_national_day"]},
    "split": {"cut_train": "2023-09-30 23:00", "cut_calibration": "2023-10-31 23:00"},
    "backtest": {"horizon": 24, "n_origins": 60, "step": 24,
                 "initial_train": 24 * 365, "window": "expanding", "m": 24},
    "models": {"point": "DirectLGBMForecaster(horizon=24)",
               "quantile": "QuantileLGBM(quantiles=(0.05, 0.50, 0.95), horizon=24)",
               "baseline": "seasonal_naive(m=24)"},
    "interval": {"method": "CQR", "alpha": 0.10, "calibration_window": "2023-10"},
    "seeds": {"lightgbm": 212, "numpy": 212},
    "refit": {"cadence": "monthly", "boundary": "2023-09-30 23:00"},
}
print(json.dumps(CONFIG, indent=2))

In [ ]:
import lightgbm, sklearn, statsmodels, scipy
from tayyar.data.load import load_demand, load_calendar, join_calendar
from tayyar.features.build import build_features
from tayyar.models.lgbm_forecaster import DirectLGBMForecaster
from tayyar.models.quantile import QuantileLGBM
from tayyar.models.conformal import CQR
from tayyar.eval.backtest import BacktestConfig, origins
from tayyar.eval.metrics import all_metrics

print("pinned at run time:",
      {"pandas": pd.__version__, "numpy": np.__version__,
       "lightgbm": lightgbm.__version__, "scikit-learn": sklearn.__version__,
       "statsmodels": statsmodels.__version__, "scipy": scipy.__version__})


def run_pipeline(conf):
    """Config in, backtest table out. No arguments hidden in the caller's scope."""
    np.random.seed(conf["seeds"]["numpy"])
    df = join_calendar(load_demand(conf["data"]["demand"]),
                       load_calendar(conf["data"]["calendar"]))
    feat = build_features(df[conf["features"]["source_columns"]])
    Xcols = [c for c in feat.columns if c != "demand_mw"]
    X, yy = feat[Xcols].astype(float), feat["demand_mw"]
    y = df["demand_mw"].astype(float)

    cfg = BacktestConfig(**conf["backtest"])
    ORIGINS = [y.index[tr.stop - 1] for tr, te in origins(y.index, cfg)]

    Xtr = X[:conf["split"]["cut_train"]].dropna(); ytr = yy[Xtr.index]
    lgbm = DirectLGBMForecaster(horizon=cfg.horizon).fit(Xtr, ytr)
    qm = QuantileLGBM(quantiles=(0.05, 0.50, 0.95), horizon=cfg.horizon).fit(Xtr, ytr)

    def one(fn, name):
        rows, per = [], []
        for o in ORIGINS:
            pos = X.index.get_loc(o); Xo = X.iloc[[pos]]
            actual = yy.iloc[pos + 1: pos + 25]
            hist = yy.iloc[pos - cfg.initial_train: pos]
            f = np.asarray(fn(Xo, pos), dtype=float)
            rows.append({"origin": o, "model": name,
                         **all_metrics(actual, f, hist, cfg.m)})
            per.append({"origin": o, "actual": actual.to_numpy(), "pred": f})
        return pd.DataFrame(rows), per

    def naive(Xo, pos):
        last = yy.iloc[pos - 23: pos + 1].to_numpy()
        return [last[h % 24] for h in range(24)]

    parts = {"LightGBM (direct)":
                 one(lambda Xo, pos: [lgbm.models_[h].predict(Xo)[0] for h in range(1, 25)],
                     "LightGBM (direct)"),
             "LightGBM (quantile q50)":
                 one(lambda Xo, pos: [qm.models_[(h, 0.50)].predict(Xo)[0]
                                      for h in range(1, 25)], "LightGBM (quantile q50)"),
             "Seasonal-naive": one(naive, "Seasonal-naive")}
    bt = pd.concat([p[0] for p in parts.values()], ignore_index=True)
    per = {k: v[1] for k, v in parts.items()}
    return dict(df=df, X=X, yy=yy, cfg=cfg, ORIGINS=ORIGINS, lgbm=lgbm, qm=qm,
                bt=bt, per=per)


art = run_pipeline(CONFIG)          # ~4 minutes: 24 + 72 LightGBM models
bt, per = art["bt"], art["per"]
MET = ["MAE", "RMSE", "MAPE_%", "WAPE_%", "MASE", "bias"]
summary = bt.groupby("model")[MET].mean().round(3).sort_values("MASE")
print()
print(summary.to_string())

In [ ]:
ref = json.load(open("../reference_results.json"))["m6"]
ref_tbl = pd.DataFrame(ref["summary"]).set_index("model")[MET].loc[summary.index]

delta = (summary - ref_tbl).abs().max().max()
print(f"largest absolute difference from reference_results.json: {delta:.6f}")
print(f"origins: run {int(bt.groupby('model').size().min())}, reference {ref['n_origins']}")

from tayyar.eval.diebold_mariano import diebold_mariano
actual = np.concatenate([p["actual"] for p in per["LightGBM (direct)"]])
P = {k: np.concatenate([p["pred"] for p in v]) for k, v in per.items()}
dm1 = diebold_mariano(actual, P["LightGBM (direct)"], P["Seasonal-naive"], h=24)
dm2 = diebold_mariano(actual, P["LightGBM (direct)"], P["LightGBM (quantile q50)"], h=24)
dm_ok = (abs(dm1["dm_stat"] - ref["dm_lgbm_vs_naive"]["dm_stat"]) < 5e-3 and
         abs(dm2["p_value"] - ref["dm_lgbm_vs_q50"]["p_value"]) < 5e-4)
print(f"DM vs naive: dm {dm1['dm_stat']} p {dm1['p_value']}   "
      f"DM vs q50: dm {dm2['dm_stat']} p {dm2['p_value']}")
print()
print("MATCH" if delta < 5e-4 and dm_ok else "MISMATCH — do not publish anything downstream")

`MATCH`, from a clean kernel, from a committed config. That is the whole point of the
task: the table in `BACKTEST.md` is not a screenshot of a good afternoon, it is the output of
a program somebody else can run.

What actually buys the reproduction, in order of how often each one is missing in practice:
the **split dates are in the config**, not typed into a cell; the **seed is inside the model
class** (`random_state=212` in `lgbm_forecaster.py` and `quantile.py`), so nobody has to
remember to pass it; the **origin schedule is derived** from `BacktestConfig` rather than
hand-rolled per notebook; and the **library versions are printed with the result**, so a
future mismatch can be attributed instead of argued about.

Now the selection step that came with the pipeline.

In [ ]:
# ---------------------------------------------------------------------------
# The selection step you inherited
#
#   "Backtest reproduces. Champion selected. Ready to promote."
#
# It runs, it uses the library function, and the model it names is genuinely the
# most accurate one in the table. It is still the wrong answer.
# ---------------------------------------------------------------------------
from tayyar.pipeline.select import champion

winner = champion(bt, metric="MASE")
print(summary.to_string())
print()
print(f"CHAMPION: {winner}")
print(f"MASE {summary.loc[winner, 'MASE']:.3f} — lowest in the table. Promoting to production.")

**The fault: selection is not `argmin`.** `pipeline.select.champion` sorts the
summary table by MASE and returns the top row. That is a correct implementation of a narrow
question — *which model was most accurate on this window?* — and the model it names, the direct
LightGBM, genuinely was. It is the wrong answer to the question actually being asked, which is
*which model do we run in production, for whom, under what constraints?*

Three things the argmin cannot see, all of them fatal in at least one of the scenarios below:

1. **The gap it selects on is not significant.** Lab 6's Diebold-Mariano put the direct model
   and the quantile model at p = 0.058. The argmin ranked them anyway, because a sort has no
   concept of a confidence interval. Selecting on an insignificant difference is selecting on
   noise, and it will pick a different winner next quarter.
2. **It ships a model that cannot be served for one of the three customers.** The direct
   LightGBM needs 44 features at serve time, including 336-hour lags and a **day-ahead
   temperature forecast**, which means an online feature store and a live weather feed. The
   regulated publication runs as a scheduled batch job with neither.
3. **It throws away the interval.** The direct model emits a point forecast. The decision
   downstream — how much spinning reserve to hold — is a quantile decision. The argmin
   optimised a column that the consumer of the forecast does not consume.

### Task 2 — three use cases, three weightings, one candidate set

**Selection is a function of the use case, not a position on a leaderboard.** The same three
models, ranked for three different customers, produce three different answers — and none of
the rankings is wrong.

The six constraints that legitimately override accuracy:

1. **Deadline and latency.** A forecast that misses the publication window has an error of
   infinity. A model that trains in four hours cannot serve a 15-minute re-run.
2. **Explainability.** A regulator, or an operator at 02:00, has to be able to interrogate the
   forecast. "Feature importance says lag-3" is not an answer to "why is it high tonight?".
3. **Cost asymmetry.** Under-forecasting load costs far more than over-forecasting it — the
   first buys emergency generation, the second buys idle spinning reserve. A symmetric MAE
   objective encodes a symmetry the business does not have. Optimise a quantile or an
   asymmetric loss and select on it.
4. **Maintenance and total cost of ownership.** One model per operating area, per horizon, per
   quantile is a pipeline somebody owns forever. Multiply by four areas before you choose.
5. **Serve-time data availability.** A feature you cannot compute at 23:00 is not a feature.
   No day-ahead temperature feed means no SARIMAX-with-temperature and no weather columns —
   whatever the backtest said.
6. **Adoption.** The winning model is the one people will actually use at 02:00 during an
   unusual event. A model the control room overrides every time it matters has zero skill in
   production regardless of its MASE.

The rule that follows: **the simplest model meeting the constraints wins, and complexity is
justified only by a DM-significant, decision-relevant gain.**

In [ ]:
from tayyar.pipeline.recommend import score, flip_conditions

# Scores are 0-1, higher is better, and every one of them is a judgement you must be
# able to defend in the decision record. maintenance = LOW burden scores HIGH.
profiles = pd.DataFrame(
    [
        # accuracy  explainability  latency  maintenance
        [0.10,      1.00,           1.00,    1.00],   # Seasonal-naive
        [0.50,      0.80,           0.95,    0.90],   # ETS
        [0.80,      0.95,           0.55,    0.60],   # SARIMAX + temperature
        [1.00,      0.45,           0.80,    0.35],   # LightGBM + CQR (per area)
        [0.90,      0.45,           0.80,    0.75],   # LightGBM + CQR (global)
    ],
    columns=["accuracy", "explainability", "latency", "maintenance"],
    index=["Seasonal-naive", "ETS", "SARIMAX + temperature",
           "LightGBM + CQR (per area)", "LightGBM + CQR (global)"])

# The hard facts that become FILTERS, not scores. A constraint you can score away is
# not a constraint.
facts = pd.DataFrame(
    {"interval": ["none", "analytic", "analytic (native)", "calibrated (CQR)",
                  "calibrated (CQR)"],
     "needs_feature_store": [False, False, False, True, True],
     "needs_temp_forecast": [False, False, True, True, True],
     "models_to_own_4_areas": [0, 4, 4, 4 * 24 * 3, 24 * 3]},
    index=profiles.index)

print(profiles.to_string())
print()
print(facts.to_string())

In [ ]:
SCENARIOS = {
    "A. Regulator (published day-ahead forecast)": {
        "weights": {"accuracy": 0.30, "explainability": 0.40,
                    "latency": 0.20, "maintenance": 0.10},
        "hard": {"needs_feature_store": False},
        "why": "published under a licence condition, must be explained on request, "
               "produced by a scheduled batch job with no online feature store",
    },
    "B. Internal dispatch (unit commitment)": {
        "weights": {"accuracy": 0.60, "explainability": 0.10,
                    "latency": 0.20, "maintenance": 0.10},
        "hard": {},
        "why": "every MW of error is money; the team owns a feature store already",
    },
    "C. Four operating areas, one analyst": {
        "weights": {"accuracy": 0.30, "explainability": 0.10,
                    "latency": 0.20, "maintenance": 0.40},
        "hard": {},
        "why": "the same person maintains all four; whatever is chosen is chosen four times",
    },
}

ranked = {}
for name, s in SCENARIOS.items():
    elig = profiles.copy()
    for col, required in s["hard"].items():
        elig = elig[facts.loc[elig.index, col] == required]
    r = score(elig, s["weights"])
    ranked[name] = r
    dropped = [m for m in profiles.index if m not in elig.index]
    print(f"{name}")
    print(f"   weights {s['weights']}")
    print(f"   why     {s['why']}")
    if dropped:
        print(f"   EXCLUDED by hard constraint: {', '.join(dropped)}")
    print(r[["accuracy", "explainability", "latency", "maintenance",
             "weighted_score"]].to_string())
    print(f"   -> champion: {r.index[0]}")
    print()

**Expected**

```
A. Regulator                 EXCLUDED by hard constraint: LightGBM + CQR (per area, global)
   SARIMAX + temperature   0.790     ETS  0.750     Seasonal-naive  0.730
B. Internal dispatch
   LightGBM + CQR (per area) 0.840   LightGBM + CQR (global) 0.820   SARIMAX 0.745
C. Four areas, one analyst
   ETS 0.780   LightGBM + CQR (global) 0.775   Seasonal-naive 0.730   SARIMAX 0.685
```

Three customers, one candidate set, three different champions — and the accuracy column never
changed. Read each one:

**A. The regulator.** The two LightGBM variants never reach the scoring step: they are removed
by a **hard filter**, not by a low score, because they cannot be served by the publication job
at all. This is the distinction to insist on in the room — *a constraint you can trade away
with a weight is not a constraint*. Among what remains, explainability at 0.40 carries
SARIMAX: a component story (trend, seasonality, a temperature coefficient with a sign and a
magnitude a regulator can question) plus native analytic intervals.

**B. Internal dispatch.** Accuracy at 0.60 and a team that already owns a feature store, so
the filter does not fire and the most accurate model wins on the merits — and, crucially, its
margin over the baseline is DM-significant (p < 0.001). Complexity bought with evidence.

**C. Four operating areas, one analyst.** Maintenance at 0.40 changes everything. The per-area
LightGBM means 4 × 24 × 3 = 288 fitted models to own, monitor and retrain; ETS means four.
Note the top two are separated by **0.005**, which is not a decision — it is a tie, and the
same rule that settled Lab 6's DM result settles it here: when two options are
indistinguishable, take the simpler or the cheaper one. Either answer is defensible; what is
not defensible is presenting 0.780 and 0.775 as if the first had won something.

In [ ]:
# Serve-time data availability is a filter too. Suppose the day-ahead temperature
# feed is down for a week — which candidates survive, and who is champion then?
elig = profiles[~facts["needs_temp_forecast"]]
r = score(elig, SCENARIOS["A. Regulator (published day-ahead forecast)"]["weights"])
print("no day-ahead temperature feed available:")
print(r[["accuracy", "explainability", "latency", "maintenance", "weighted_score"]]
      .to_string())
print(f"-> champion: {r.index[0]}  (SARIMAX + temperature is not a candidate without "
      f"a temperature forecast)")

### Task 3 — the champion changes, and you must say what changed it

A recommendation that does not state its own flip-conditions is not a recommendation, it is a
preference. `flip_conditions` answers one question precisely: **how much would have to move,
and in which column, for the runner-up to win?** If the answer is "0.02 of explainability",
the decision is a coin toss dressed as analysis and you must say so.

In [ ]:
rows = []
for name, s in SCENARIOS.items():
    r = ranked[name]
    rows.append({"scenario": name.split(".")[0],
                 "champion": r.index[0],
                 "score": r.weighted_score.iloc[0],
                 "runner-up": r.index[1],
                 "gap": round(float(r.weighted_score.iloc[0] - r.weighted_score.iloc[1]), 3),
                 "decisive constraint": {"A": "explainability + serve-time (hard filter)",
                                         "B": "accuracy, DM-significant",
                                         "C": "maintenance"}[name.split(".")[0]]})
print(pd.DataFrame(rows).to_string(index=False))
print()
for name, s in SCENARIOS.items():
    print(name)
    for line in flip_conditions(ranked[name], s["weights"]):
        print("   " + line)
    print()

Read the gaps before you read the champions. Scenario A is decided by 0.040, and only
after two candidates were removed by a **hard filter** that no weighting could have overridden.
Scenario B picks a *family* decisively — the best LightGBM stands 0.095 clear of the best
non-LightGBM candidate, and that margin is backed by a Diebold-Mariano result — but the choice
*within* the family, per-area against global, is 0.020 and is not decided by this table at all.
Scenario C is decided by **0.005**, which is to say it is not decided: `flip_conditions` prints
a required move of 0.01 in maintenance, well inside the precision of the judgements that
produced the table.

That is the honest output of this task, and it belongs in the record verbatim:

- **A flipped on explainability plus serve-time availability.** The LightGBM candidates were
  never eligible; among those that were, weighting explainability at 0.40 chose SARIMAX.
- **B flipped on accuracy**, and the accuracy gap over the baseline is DM-significant.
- **C flipped on maintenance.** 288 models against 4. The two leaders are tied; report the tie.

The general rule this makes concrete: **the simplest model that meets the constraints wins.**
Complexity has to be bought with a DM-significant, decision-relevant gain, and in two of these
three scenarios it cannot be.

### Task 4 — the brief, and what a control-room engineer does with it

The forecast is finished. The **communication** is not, and it is where forecasts are most
often rejected. Lead with the decision, not the model.

In [ ]:
from tayyar.pipeline.brief import reserve_brief

X_, yy_, qm_, ORIGINS_ = art["X"], art["yy"], art["qm"], art["ORIGINS"]
TAUS = (0.05, 0.50, 0.95)

# Calibrate on October, exactly as the config says, then issue the last origin's day.
Xca = X_[CONFIG["split"]["cut_train"]:CONFIG["split"]["cut_calibration"]]
yca = yy_[Xca.index]
rows_ca = list(range(0, len(Xca) - 24, 24))
Yca = np.array([yca.iloc[i + 1: i + 25].to_numpy() for i in rows_ca])
Pca = {t: np.column_stack([qm_.models_[(h, t)].predict(Xca.iloc[rows_ca])
                           for h in range(1, 25)]) for t in (0.05, 0.95)}
cqr = CQR(alpha=CONFIG["interval"]["alpha"]).calibrate(
    Yca.ravel(), Pca[0.05].ravel(), Pca[0.95].ravel())

o = ORIGINS_[-1]
pos = X_.index.get_loc(o); Xo = X_.iloc[[pos]]
q = {t: np.array([qm_.models_[(h, t)].predict(Xo)[0] for h in range(1, 25)]) for t in TAUS}
lo, hi = cqr.interval(q[0.05], q[0.95])
fc = pd.DataFrame({"q05": lo, "q50": np.clip(q[0.50], lo, hi), "q95": hi},
                  index=pd.date_range(o + pd.Timedelta(1, "h"), periods=24, freq="h"))

brief = reserve_brief(
    fc, coverage=0.90, n_origins=60,
    weakness=("Eid and public holidays. This backtest window (Nov-Dec) contains none, "
              "and at the 2023 Eid al-Fitr the same model's error roughly doubles. "
              "Also the hottest afternoons: marginal coverage is 0.90 but only ~0.85 "
              "on the 12:00-18:00 block."),
    fallback=("seasonal-naive plus the on-call analyst's uplift, with the interval taken "
              "from the last 30 days' empirical errors. Trigger it on any Eid day, on any "
              "day the temperature feed is missing, or when rolling 7-day MASE exceeds 0.60."),
    issued=f"{o:%Y-%m-%d %H:%M}")
print(brief)

In [ ]:
peak_i = fc.q50.idxmax()
peak, p95 = fc.q50.max(), fc.q95.max()

print("THE JARGON VERSION — every number in it is correct, and not one of them is a decision")
print(f"  The p95 is {p95:,.0f} MW, MASE is 0.455, marginal coverage is 0.90, and the "
      f"mean pinball loss is 201 MW.")
print()
print("THE DECISION VERSION — same numbers, addressed to the person who has to act")
print(f"  Cover demand up to {p95:,.0f} MW tomorrow: hold {p95 - peak:,.0f} MW above the "
      f"expected peak of")
print(f"  {peak:,.0f} MW at {peak_i:%H:%M}. That gap is the honest uncertainty, and it "
      f"widens on hot days;")
print(f"  it covers all but about 5% of afternoons. It is least reliable on Eid days, "
      f"which this")
print(f"  evidence does not cover at all — on those, fall back to the analyst uplift.")

**Critique it the way the engineer will.**

The template gets the *order* right, and the order is most of the battle: the recommended
reserve appears before the model is mentioned, and the model is never mentioned at all. It
gets three more things right that are usually missing. It states the **coverage the interval
actually achieved** over 60 backtested days rather than the nominal figure it was designed for.
It names **where the forecast is weakest** in the same breath as the recommendation, not in an
appendix. And it names a **fallback**, with triggers, so the reader knows what to do at 02:00
when it is wrong — which is the question the third paragraph of every brief should answer and
almost none do.

What it still does not do, and what the engineer will ask for: it gives one reserve number for
the day when the uncertainty is not flat across the day, and it quotes marginal coverage when
the peak-hour coverage is the number that matters for a peak-hour decision. Both belong in the
next revision, and both are already measured in Lab 6.

One more thing to fix, and it is the kind of thing that gets a brief sent back: the word
**reserve** is used for two different quantities. The template calls the 1,199 MW *margin* the
"recommended spinning reserve"; a control room may equally read "reserve" as the level to cover
to, 33,323 MW. Both readings are defensible from the words alone, and the difference is 32,000
MW. Decision language has to be unambiguous about the unit and the datum — say "hold 1,199 MW
above the expected peak" or "cover to 33,323 MW", never the bare noun.

**The two versions, side by side.** Every number in the jargon line is correct and none of them
is a decision. "The p95 is 33,323 MW" is a fact about a distribution; "hold 1,199 MW above the
expected peak of 32,124 MW at 18:00" is an instruction somebody can execute. "MASE is 0.455"
answers a question nobody in the control room asked; "this covers all but about 5% of
afternoons" answers the one they did. The decision version carries no less information — it
carries the same information, addressed to the person who has to act on it, plus the one
sentence the jargon version omits entirely: **where it fails and what to do then.**

Note the size of that gap and why it is small. 31 December is a mild winter day, peaking at
32,124 MW; the interval is correspondingly narrow. Across the sixty backtested days the mean
band width was 3,213 MW, and on a July afternoon it is wider again. **That variation is the
interval working**, not noise in it — a constant-width band would be too wide tonight and far
too narrow in August. It is also why the brief says the uncertainty is widest on hot days
rather than quoting one number as if it held all year.

### Task 5 — `DECISION_RECORD.md`

Seven sections, in this order. The order is the argument.

**1. The decision and its cost function.** *How much spinning reserve to hold for each hour of
tomorrow.* Under-forecasting buys emergency generation at a large multiple of the marginal
cost, and in the tail buys load shedding; over-forecasting buys idle capacity at the marginal
cost of holding it. The costs are asymmetric, so the object being decided is a **quantile**,
not a mean, and the model is selected on that basis.

**2. The candidates.** Seasonal-naive, ETS, SARIMAX + temperature, LightGBM + CQR (per area),
LightGBM + CQR (global). With the profile table.

**3. The constraints applied**, hard filters separated from weighted criteria — publication
deadline, explainability to the regulator, cost asymmetry, maintenance across four areas,
serve-time data availability (feature store, day-ahead temperature feed), and adoption by the
control room.

**4. The stated weights**, per scenario, written down before the scores were computed.

**5. The recommendation and its evidence.** The champion per scenario, with the Lab 6
Diebold-Mariano results attached: significant against the baseline (dm = -6.75, p < 0.001),
**not** significant between the two LightGBM variants (p = 0.058) — which is why the quantile
model is preferred over the marginally more accurate direct model in scenario B, and why
scenario C's 0.005 gap is reported as a tie.

**6. The flip-conditions.** Verbatim from `flip_conditions`, per scenario, plus the operational
ones: no day-ahead temperature feed removes SARIMAX + temperature; a fifth operating area
moves scenario C further towards the global model; a regulator accepting a documented
importance analysis re-admits LightGBM to scenario A.

**7. The monitoring contract**, which is the part that makes the recommendation survive
contact with production:

| what | how often | threshold | action |
|---|---|---|---|
| rolling MASE, 7-day and 28-day | weekly | 28-day MASE > 0.60 | investigate; > 0.77 (the baseline's own score) means fall back |
| empirical coverage of the 90% band | weekly | outside 0.85-0.95 for two consecutive weeks | re-estimate q̂ on the last 30 days |
| conditional coverage on the 12:00-18:00 block | weekly | below 0.85 | widen the peak-hour band; consider adaptive conformal |
| feature drift (temperature, weekly mean level) | weekly | new maximum outside the training range | expect extrapolation failure; trees cannot forecast a record high |
| retrain | monthly, and on any threshold breach | — | full refit, then re-run this backtest before promotion |
| champion / challenger | continuous | challenger beats champion on a DM test at 5% over 60 origins | promote; otherwise keep the incumbent |

**Say what the recommendation cannot do.** This backtest window contains no Eid, no summer
heatwave and no record peak. That absence is a finding, and naming it is what earns adoption:
a report that names the Eid weeks and ships a fallback should outscore a silent, marginally
sharper submission, because the second one will be overridden the first time it is wrong and
then ignored forever after. Candour is not a weakness in a recommendation; it is the thing
that makes it usable at 02:00.

### Mini-quiz — answers

1. **Is lowest MASE always right?** No. The deadline, explainability, the asymmetry between
   the cost of under- and over-forecasting, maintenance burden, serve-time data availability
   and adoption can each override it — and a gap that fails a Diebold-Mariano test is not a
   reason to override anything.
2. **Covariates plus a regulator demanding explainability?** **SARIMAX with exogenous
   regressors.** It has a component story you can narrate — trend, seasonality, the
   temperature coefficient with a sign and a magnitude — and it produces native analytic
   intervals. LightGBM offers feature importance, which is an association, not an explanation.
3. **What else must a recommendation include?** Its own **limits** — the flip-conditions that
   would change the answer, and the regimes the evidence does not cover — and a **monitoring
   plan** with thresholds and a fallback. A recommendation without limits is an advertisement.
4. **Why a quantile for reserve sizing?** Because the decision is a risk decision. Reserve is
   sized so that demand exceeds supply with acceptably small probability; that is a quantile,
   and the cost of being short is not the cost of being long. A point forecast forces the
   consumer to invent the quantile themselves, usually badly.
5. **What makes it reproducible?** Fixed splits and fixed seeds, pinned dependency versions,
   and a config-driven pipeline that runs end to end from a committed file — so the table in
   the report can be regenerated by somebody who was not in the room.

### Task 6 — commit

```
git add -A && git commit -m "feat(pipeline): constraint-weighted selection, reserve brief and decision record"
```

The two files that leave this course are `BACKTEST.md` (the evidence) and
`DECISION_RECORD.md` (the decision). The notebooks are the working; those two are the work.

### Instructor notes

This lab folds directly into the capstone assembly. Whatever the room writes here is the
skeleton of their submission, so do not let it become a discussion — make them type.

- **Branch `sim-leaderboard`** ships the Task 1 fault alone: the pipeline reproduces, the
  argmin selects the direct LightGBM, and the deployment note says "publication job, batch, no
  feature store". Nothing errors. The fix is the constraint filter and a re-run of
  `recommend.score`, and the room should reach it without being told which line is wrong.
- **The communication drill** (10 minutes, at Task 4). Each pair rewrites the jargon summary
  into exactly **two** sentences of decision language — a hard limit; three sentences is a
  hedge. Then a participant plays the control-room engineer and pushes back with the three
  questions that always come: *what do I do differently because of this?*, *how wrong has it
  been lately?*, and *what do I do when it is wrong tonight?* A brief that cannot answer the
  third one gets sent back.
- **The walkthrough to tell before Task 2.** A utility promoted a gradient-boosted model on
  MASE alone. It was, on the evidence, the most accurate model they had. During an unusual load
  event the control-room engineer could not interrogate why it was forecasting what it was
  forecasting, so he overrode it — and then kept overriding it, and the model's realised skill
  in production went to zero. **The failure was the selection, not the model.** Nobody had
  weighted explainability, and nobody had asked the person who would have to act on it at
  02:00. Ask the room whose fault that was; the useful answer is that it was the selection
  criterion's.
- **Let them argue about the weights.** The weights are the only genuinely subjective input in
  this lab, and the argument is the point: a decision record exists so that a future reader can
  see what was assumed and challenge it. Time-box it to five minutes and require the numbers to
  be written down *before* the scores are computed.

**Troubleshooting**

| symptom | cause |
|---|---|
| the same champion in every scenario | the weights are barely different, or the hard filters are not being applied. Print the eligible set per scenario. |
| pipeline numbers differ from Lab 6 | a split date typed into a cell instead of read from the config, an unseeded model, or an unpinned LightGBM. Print the versions with the result. |
| `recommend` returns no candidate | the hard constraints are collectively unsatisfiable. That is a real finding — report it as "no candidate meets these constraints" and negotiate one of them, rather than quietly relaxing it. |
| the brief reads as jargon | rewrite it in reserve-margin terms, with the weakness named in the same paragraph as the recommendation. |
| flip_conditions prints nothing useful | the gap exceeds any single criterion's weight, i.e. no one-column move flips it. Say that — it is a strong result. |

**If a cohort is behind**, reduce Task 2 to two scenarios and demo the third from the front.
**Never drop the decision record.** A model without one is not deliverable, and the record is
what the capstone is actually assessed on.